## Exercice 1: Tokenisation avec BERT

In [1]:
import torch

# Installer les bibliothèques transformers
!pip install transformers torch

In [2]:
from transformers import BertTokenizer

# Charger le tokenizer BERT (bert-base-uncased)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Choisir une phrase d'exemple
sentence = "Le BERT est un modèle de langage puissant."

# Tokeniser la phrase et visualiser comment BERT la décompose
tokens = tokenizer.tokenize(sentence)
print("Tokens bruts:", tokens)

# Préparer la phrase avec des tokens spéciaux, le padding et la troncation pour l'entrée du modèle
encoded_input = tokenizer(sentence, padding=True, truncation=True, return_tensors='pt')

print("\nInput IDs (IDs numériques des tokens):", encoded_input['input_ids'])
print("Attention Mask (masque d'attention):", encoded_input['attention_mask'])

# Décoder les IDs pour voir les tokens spéciaux ajoutés par BERT
decoded_tokens = tokenizer.convert_ids_to_tokens(encoded_input['input_ids'][0])
print("\nTokens avec tokens spéciaux:", decoded_tokens)

print("\nObservation des tokens spéciaux ajoutés par BERT: [CLS] au début, [SEP] à la fin de la phrase.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokens bruts: ['le', 'bert', 'est', 'un', 'model', '##e', 'de', 'lang', '##age', 'pu', '##issa', '##nt', '.']

Input IDs (IDs numériques des tokens): tensor([[  101,  3393, 14324,  9765,  4895,  2944,  2063,  2139, 11374,  4270,
         16405, 21205,  3372,  1012,   102]])
Attention Mask (masque d'attention): tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Tokens avec tokens spéciaux: ['[CLS]', 'le', 'bert', 'est', 'un', 'model', '##e', 'de', 'lang', '##age', 'pu', '##issa', '##nt', '.', '[SEP]']

Observation des tokens spéciaux ajoutés par BERT: [CLS] au début, [SEP] à la fin de la phrase.


## Exercice 2: Analyse de Sentiment avec un Pipeline BERT

In [3]:
from transformers import pipeline

# Créer un pipeline d'analyse de sentiment
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Fournir une phrase d'exemple
sample_sentence = "J'adore ce film, il est fantastique !"
another_sentence = "Ce n'était pas très bon, je suis déçu."

# Utiliser le pipeline pour prédire le sentiment
result_positive = sentiment_pipeline(sample_sentence)
result_negative = sentiment_pipeline(another_sentence)

# Examiner l'étiquette prédite et le score de confiance
print(f"Phrase: '{sample_sentence}'\nRésultat: {result_positive}")
print(f"\nPhrase: '{another_sentence}'\nRésultat: {result_negative}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Phrase: 'J'adore ce film, il est fantastique !'
Résultat: [{'label': 'POSITIVE', 'score': 0.9991869330406189}]

Phrase: 'Ce n'était pas très bon, je suis déçu.'
Résultat: [{'label': 'NEGATIVE', 'score': 0.6771425604820251}]


## Exercice 3: Construire un Analyseur de Sentiment Personnalisé

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

class BERTSentimentAnalyzer:
    def __init__(self, model_name="distilbert-base-uncased-finetuned-sst-2-english"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        # Obtenir les mappings d'ID à Label et vice versa
        self.id2label = self.model.config.id2label
        self.label2id = self.model.config.label2id

    def preprocess(self, text):
        # Nettoyage, tokenisation et préparation des tenseurs
        encoded_input = self.tokenizer(text, return_tensors='pt', padding=True, truncation=True)
        return {k: v.to(self.device) for k, v in encoded_input.items()}

    def predict(self, text):
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)

        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)

        # Obtenir l'index de la classe avec la plus haute probabilité
        predicted_class_id = torch.argmax(probabilities, dim=-1).item()
        predicted_label = self.id2label[predicted_class_id]
        confidence_score = probabilities[0][predicted_class_id].item()

        return {"label": predicted_label, "score": confidence_score}

# Instancier l'analyseur
analyzer = BERTSentimentAnalyzer()

# Tester votre analyseur avec différents textes d'exemple
text1 = "J'ai vraiment apprécié cette expérience, c'était incroyable !"
text2 = "Je ne suis pas du tout satisfait de ce service. Horrible."
text3 = "Le temps est agréable aujourd'hui."

print(f"Analyse de sentiment pour '{text1}': {analyzer.predict(text1)}")
print(f"Analyse de sentiment pour '{text2}': {analyzer.predict(text2)}")
print(f"Analyse de sentiment pour '{text3}': {analyzer.predict(text3)}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Analyse de sentiment pour 'J'ai vraiment apprécié cette expérience, c'était incroyable !': {'label': 'POSITIVE', 'score': 0.8543674945831299}
Analyse de sentiment pour 'Je ne suis pas du tout satisfait de ce service. Horrible.': {'label': 'NEGATIVE', 'score': 0.9994980096817017}
Analyse de sentiment pour 'Le temps est agréable aujourd'hui.': {'label': 'POSITIVE', 'score': 0.9818747043609619}


## Exercice 4: Comprendre BERT pour la Reconnaissance d'Entités Nommées (REN)

In [5]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline # Pour une implémentation simplifiée si on le souhaite, mais ici on fait personnalisé
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name="dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.id2label = self.model.config.id2label

    def recognize_entities(self, text):
        # Tokeniser le texte
        tokens = self.tokenizer.tokenize(self.tokenizer.decode(self.tokenizer.encode(text)))
        # Obtenir les IDs d'entrée
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)

        predictions = torch.argmax(outputs.logits, dim=2).squeeze().tolist()

        # Mapper les prédictions de tokens aux étiquettes
        entities = []
        current_entity = {"word": [], "entity": None}

        for token_id, prediction_id in zip(inputs['input_ids'].squeeze().tolist(), predictions):
            token = self.tokenizer.convert_ids_to_tokens(token_id)
            label = self.id2label[prediction_id]

            if token.startswith('##'): # Gérer les sous-mots BERT
                token = token[2:]

            # Ignorer les tokens spéciaux [CLS] et [SEP]
            if token in [self.tokenizer.cls_token, self.tokenizer.sep_token]:
                if current_entity["word"]:
                    entities.append({"word": " ".join(current_entity["word"]), "entity": current_entity["entity"]})
                    current_entity = {"word": [], "entity": None}
                continue

            if label.startswith('B-'): # Début d'une nouvelle entité
                if current_entity["word"]:
                    entities.append({"word": " ".join(current_entity["word"]), "entity": current_entity["entity"]})
                current_entity["word"] = [token]
                current_entity["entity"] = label[2:]
            elif label.startswith('I-') and current_entity["entity"] == label[2:]:
                current_entity["word"].append(token)
            elif label == 'O': # En dehors d'une entité
                if current_entity["word"]:
                    entities.append({"word": " ".join(current_entity["word"]), "entity": current_entity["entity"]})
                    current_entity = {"word": [], "entity": None}
                # On pourrait ajouter les tokens 'O' aussi si on voulait tout garder
            else: # Cas où 'I-' ne correspond pas à 'B-' précédent ou autre cas de transition
                if current_entity["word"]:
                    entities.append({"word": " ".join(current_entity["word"]), "entity": current_entity["entity"]})
                    current_entity = {"word": [], "entity": None}
                # Si le label est I- mais pas de B- avant, ou un autre type d'entité
                if label != 'O': # Si c'est un I- ou B- isolé, le traiter comme une nouvelle entité d'un mot
                    current_entity["word"] = [token]
                    current_entity["entity"] = label[2:]

        if current_entity["word"]:
            entities.append({"word": " ".join(current_entity["word"]), "entity": current_entity["entity"]})

        return entities

# Instancier le Recognizer
ner_recognizer = BERTNamedEntityRecognizer()

# Tester avec un texte d'exemple
sample_text_ner = "Barack Obama est né à Honolulu et a étudié à l'Université Harvard. Il a rencontré Angela Merkel à Berlin."
entities = ner_recognizer.recognize_entities(sample_text_ner)
print(f"Texte: '{sample_text_ner}'")
print("Entités reconnues:")
for entity in entities:
    print(f" - Mot: '{entity['word']}', Entité: {entity['entity']}")

sample_text_ner_2 = "Apple Inc. est une entreprise américaine de technologie fondée par Steve Jobs, Steve Wozniak et Ronald Wayne."
entities_2 = ner_recognizer.recognize_entities(sample_text_ner_2)
print(f"\nTexte: '{sample_text_ner_2}'")
print("Entités reconnues:")
for entity in entities_2:
    print(f" - Mot: '{entity['word']}', Entité: {entity['entity']}")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Texte: 'Barack Obama est né à Honolulu et a étudié à l'Université Harvard. Il a rencontré Angela Merkel à Berlin.'
Entités reconnues:
 - Mot: 'Barack Obama', Entité: PER
 - Mot: 'Honolulu', Entité: LOC
 - Mot: 'l', Entité: ORG
 - Mot: 'Université Harvard', Entité: ORG
 - Mot: 'Angela Me rk el', Entité: PER
 - Mot: 'Berlin', Entité: LOC

Texte: 'Apple Inc. est une entreprise américaine de technologie fondée par Steve Jobs, Steve Wozniak et Ronald Wayne.'
Entités reconnues:
 - Mot: 'Apple Inc', Entité: ORG
 - Mot: 'Steve Job s', Entité: PER
 - Mot: 'Steve W oz nia k', Entité: PER
 - Mot: 'Ronald Wayne', Entité: PER


## Exercice 5: Comparaison entre BERT et GPT

Voici un tableau comparatif entre les modèles BERT et GPT:

| Caractéristique       | BERT (Bidirectional Encoder Representations from Transformers) | GPT (Generative Pre-trained Transformer) |
|-----------------------|---------------------------------------------------------------|------------------------------------------|
| **Architecture**      | Encodeur (Transformer Encoder)                                | Décodeur (Transformer Decoder)           |
| **Directionnalité**   | Bidirectionnel (lit le texte dans les deux sens)              | Unidirectionnel (lit le texte de gauche à droite) |
| **Objectif Principal**| Compréhension du langage (encoder le contexte d'un mot)      | Génération de texte (prédire le prochain mot) |
| **Cas d'Utilisation Communs** | Analyse de sentiment, REN, QA, Résumé, Classification de texte | Génération de texte, Traduction, Chatbots, Résumé (génératif) |
| **Strengths**         | Excellente compréhension contextuelle, performant sur les tâches d'analyse et de classification. | Excellente capacité de génération de texte fluide et cohérent, performant pour les tâches créatives. |
| **Weaknesses**        | Moins adapté à la génération de texte longue, pas de capacité de génération naturelle. | Moins performant pour les tâches nécessitant une compréhension bidirectionnelle profonde du contexte, peut parfois "halluciner" (générer des informations fausses). |
| **Philosophie**       | Comprendre le sens d'un texte par son contexte bidirectionnel | Générer la suite la plus probable d'un texte par une approche unidirectionnelle |

**Réflexion sur les différences et similarités:**

*   **Similarités:** Les deux modèles sont basés sur l'architecture Transformer, ce qui leur permet de gérer les dépendances à longue portée dans le texte et d'être entraînés sur de vastes corpus de données.
*   **Différences:** La distinction fondamentale réside dans leur architecture (encodeur vs décodeur) et leur objectif principal (compréhension vs génération). BERT est conçu pour *comprendre* le contexte d'un mot en le lisant des deux côtés, ce qui est crucial pour les tâches d'analyse. GPT, quant à lui, est conçu pour *générer* du texte en prédisant le mot suivant, ce qui le rend idéal pour la création de contenu.

## Exercice 6: Applications de BERT dans les systèmes RAG (Retrieval-Augmented Generation)

### Qu'est-ce que la Génération Augmentée par Récupération (RAG) ?

La Génération Augmentée par Récupération (RAG) est une architecture de modèle qui combine la puissance des modèles de langage génératifs (comme GPT) avec des systèmes de récupération d'informations (comme ceux utilisant BERT). L'idée est que, au lieu de générer une réponse uniquement basée sur ce qu'il a appris pendant son entraînement, le modèle de langage peut d'abord récupérer des informations pertinentes à partir d'une base de connaissances externe, puis utiliser ces informations pour générer une réponse plus précise, pertinente et factuellement fondée. Cela permet aux modèles de rester à jour avec les informations les plus récentes et de réduire les "hallucinations".

### Rôle de BERT dans le composant de récupération

BERT joue un rôle crucial dans la phase de *récupération* d'un système RAG. Sa capacité à comprendre le contexte et la sémantique du texte le rend idéal pour trouver des documents ou des passages pertinents qui peuvent aider le modèle génératif. Plus précisément, BERT est souvent utilisé pour:

1.  **Encoder la requête (query):** Lorsque l'utilisateur pose une question, BERT transforme cette question en un vecteur numérique (un *embedding* ou *incorporation*). Cet embedding capture le sens sémantique de la requête.
2.  **Encoder les documents/passages:** De la même manière, tous les documents ou passages de la base de connaissances externe (corpus) sont pré-encodés par BERT en embeddings. Ces embeddings sont stockés dans une base de données vectorielle.

### Comment BERT génère des embeddings pour les documents et les requêtes

Lorsque BERT traite un texte (que ce soit une requête ou un document), il génère une représentation contextuelle pour chaque mot. Le token `[CLS]` (Class Token) au début de l'entrée de BERT est souvent utilisé comme une représentation agrégée de l'ensemble de la phrase ou du document. Le vecteur de sortie correspondant au token `[CLS]` est extrait et sert d'*embedding sémantique* pour l'ensemble du texte. Ces embeddings sont des points dans un espace multidimensionnel où les textes ayant un sens similaire sont situés plus près les uns des autres.

### Utilisation d'une base de données vectorielle pour faire correspondre les requêtes aux documents pertinents

Une fois que la requête de l'utilisateur et les documents sont représentés sous forme d'embeddings vectoriels, une *base de données vectorielle* (par exemple, FAISS, Pinecone, Weaviate) est utilisée pour trouver rapidement les documents les plus pertinents. Le processus est le suivant:

1.  **Indexation:** Les embeddings des documents sont indexés dans la base de données vectorielle.
2.  **Recherche de similarité:** Lorsque l'embedding de la requête est généré, la base de données vectorielle effectue une recherche de similarité (par exemple, en utilisant la similarité cosinus ou la distance euclidienne) pour trouver les embeddings de documents les plus proches de l'embedding de la requête.
3.  **Récupération:** Les documents (ou passages) correspondant à ces embeddings les plus proches sont alors récupérés. Ce sont les informations pertinentes que le modèle génératif utilisera.

### Exemple de collaboration entre BERT et un modèle génératif (comme GPT) dans un système RAG

**Scénario:** Un utilisateur pose la question: "Quelle est la capitale de la France et pourquoi est-elle célèbre ?"

1.  **Phase de Récupération (pilotée par BERT):**
    *   **Encodage de la requête:** La question de l'utilisateur est passée à travers un modèle BERT (ou Sentence-BERT, une variante optimisée pour les embeddings de phrases) pour générer un embedding vectoriel de la requête.
    *   **Recherche dans la base vectorielle:** Cet embedding de requête est ensuite utilisé pour interroger une base de données vectorielle contenant des embeddings de millions de documents (articles encyclopédiques, pages web, etc.).
    *   **Récupération de documents:** La base de données vectorielle renvoie les N documents les plus pertinents. Par exemple, elle pourrait récupérer des articles sur "Paris", "Histoire de la France", "Monuments de Paris", etc.

2.  **Phase de Génération (pilotée par GPT):**
    *   **Contexte étendu:** Les documents récupérés sont ensuite transmis au modèle génératif (GPT) *avec* la requête originale de l'utilisateur. GPT reçoit alors un prompt de ce type: "Voici des informations contextuelles: [documents récupérés]. En te basant sur ces informations, réponds à la question: 'Quelle est la capitale de la France et pourquoi est-elle célèbre ?'"
    *   **Génération de la réponse:** GPT utilise ces informations contextuelles pour formuler une réponse détaillée et factuellement correcte, par exemple: "La capitale de la France est Paris. Elle est célèbre pour sa richesse historique, ses monuments emblématiques comme la Tour Eiffel et le Louvre, sa gastronomie, sa mode et son influence culturelle mondiale."

Dans cet exemple, BERT a permis de trouver les informations pertinentes qui ont ensuite guidé GPT à générer une réponse bien informée, au lieu de s'appuyer uniquement sur ses connaissances internes potentiellement obsolètes ou incomplètes.